# 02 — Modeling and Optimization

## Objective

This notebook develops and evaluates supervised classification models for predicting visitor conversion.

The objectives are to:

* establish a reproducible logistic-regression baseline using the original predictors;
* evaluate model performance using F1-score, precision, recall, and the confusion matrix;
* estimate generalization performance using stratified cross-validation;
* diagnose potential underfitting, overfitting, and class-imbalance effects;
* optimize logistic regression using a focused set of justified hyperparameters;
* evaluate targeted feature engineering only where supported by the exploratory findings;
* compare the optimized linear approach with one justified nonlinear challenger;
* select the final modeling strategy based on predictive performance, generalization, stability, and complexity.

## Methodology

The competition's primary evaluation metric is the **F1-score**, so F1 will be used for model comparison, cross-validation, and hyperparameter optimization. Precision, recall, and confusion matrices will provide complementary information about the types of classification errors made by each model.

The internal test set created in the previous notebook remains reserved for final internal evaluation. Model development and optimization will be performed using the training portion of the labeled dataset, with **stratified cross-validation** used to estimate generalization performance.

Preprocessing will be integrated directly into scikit-learn pipelines so that learned transformations are fitted independently within each training fold during cross-validation, preventing data leakage.

The modeling strategy follows an incremental approach:

**Baseline → evaluation → focused optimization → targeted feature engineering → nonlinear challenger → model comparison**

Additional complexity will be retained only when it provides a meaningful and stable improvement in predictive performance.

## Notebook Structure

1. Imports, configuration, and data preparation
2. Logistic-regression baseline
3. Baseline evaluation and cross-validation
4. Logistic-regression optimization
5. Targeted feature engineering
6. Nonlinear challenger
7. Model comparison and selection
8. Conclusion and next steps


### 1. Imports, configuration, and data preparation

In [ ]:
# Imports and reproducibility
# ---------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    cross_val_predict,
    train_test_split,
)

from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    PolynomialFeatures,
    StandardScaler,
)


from sklearn.pipeline import Pipeline







# Reproducibility settings
# ---------------------------------------------------------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


# Project paths
# ---------------------------------------------------------------------------

TRAIN_DATA_PATH = Path("../data/raw/conversion_data_train.csv")
FIGURES_PATH = Path("../outputs/figures")
FIGURES_PATH.mkdir(parents=True, exist_ok=True)


# Package versions and configuration
# ---------------------------------------------------------------------------

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Random state: {RANDOM_STATE}")
print(f"Training data path: {TRAIN_DATA_PATH}")

### 1. Imports, Configuration, and Data Preparation

The labeled dataset is reloaded so that this notebook can be executed independently from the exploratory analysis notebook.

The two implausible age observations identified during data-quality assessment are removed using the same cleaning rule established previously. The cleaned data is then divided using the same stratified 80/20 split and random state.

This reproduces the model-development and internal test sets established in Notebook 01 without relying on notebook state or manually exported intermediate datasets.


In [ ]:
# Load and reproduce the cleaned labeled dataset
# ---------------------------------------------------------------------------

train_data = pd.read_csv(TRAIN_DATA_PATH)

initial_size = len(train_data)

train_data = (
    train_data.loc[train_data["age"] <= 80]
    .reset_index(drop=True)
)

print(f"Original observations: {initial_size:,}")
print(f"Cleaned observations: {len(train_data):,}")
print(f"Rows removed: {initial_size - len(train_data):,}")

In [ ]:
# Features, target, and stratified split
# ---------------------------------------------------------------------------

X = train_data.drop(columns="converted")
y = train_data["converted"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training observations: {len(X_train):,}")
print(f"Internal test observations: {len(X_test):,}")
print(f"Training conversion rate: {y_train.mean():.3%}")
print(f"Internal test conversion rate: {y_test.mean():.3%}")

## 2. Logistic-Regression Baseline

A logistic-regression classifier is used as the initial supervised-learning baseline.

Logistic regression provides a strong reference model because it is computationally efficient, interpretable, and appropriate for binary classification. Establishing an untuned baseline also provides a clear benchmark against which subsequent optimization, feature engineering, and nonlinear modeling can be evaluated.

The baseline uses the five original predictors without engineered features.

Preprocessing is integrated directly into the model pipeline:

* `age` and `total_pages_visited` are standardized;
* `country` and `source` are one-hot encoded;
* `new_user` is retained as a binary indicator.

The classifier initially uses its default decision threshold and no class weighting. These choices will only be reconsidered after baseline performance has been evaluated.


In [ ]:
# Baseline feature groups
# ---------------------------------------------------------------------------

numerical_features = [
    "age",
    "total_pages_visited",
]

categorical_features = [
    "country",
    "source",
]

binary_features = [
    "new_user",
]


# Baseline preprocessing
# ---------------------------------------------------------------------------

baseline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            categorical_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
)

In [ ]:
# Logistic-regression baseline pipeline
# ---------------------------------------------------------------------------

baseline_model = Pipeline(
    steps=[
        (
            "preprocessor",
            baseline_preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

baseline_model

In [ ]:
# Fit baseline model
# ---------------------------------------------------------------------------

baseline_model.fit(X_train, y_train)

print("Baseline logistic regression fitted successfully.")

In [ ]:
# Baseline predictions
# ---------------------------------------------------------------------------

y_train_pred = baseline_model.predict(X_train)
y_test_pred = baseline_model.predict(X_test)


# Baseline performance
# ---------------------------------------------------------------------------

baseline_metrics = pd.DataFrame(
    {
        "Train": {
            "F1": f1_score(y_train, y_train_pred),
            "Precision": precision_score(y_train, y_train_pred),
            "Recall": recall_score(y_train, y_train_pred),
        },
        "Internal test": {
            "F1": f1_score(y_test, y_test_pred),
            "Precision": precision_score(y_test, y_test_pred),
            "Recall": recall_score(y_test, y_test_pred),
        },
    }
)

baseline_metrics.round(4)

### Initial baseline results

The untuned logistic-regression baseline achieves an **F1-score of 0.7637 on the training set** and **0.7602 on the internal test set**.

The small difference between training and internal-test performance suggests that the baseline generalizes well and shows no substantial evidence of overfitting.

On the internal test set, precision (**0.8641**) is higher than recall (**0.6786**). The model is therefore relatively conservative when predicting conversion: most visitors classified as converters are correctly identified, but a meaningful proportion of actual converters is missed.

These results provide a strong initial benchmark. Before considering optimization, the classification errors and cross-validation stability should be examined in more detail.


In [ ]:
# Baseline confusion matrix
# ---------------------------------------------------------------------------

baseline_confusion_matrix = confusion_matrix(
    y_test,
    y_test_pred,
)

tn, fp, fn, tp = baseline_confusion_matrix.ravel()

print("Internal test confusion matrix:")
print(baseline_confusion_matrix)

print(f"\nTrue negatives:  {tn:,}")
print(f"False positives: {fp:,}")
print(f"False negatives: {fn:,}")
print(f"True positives:  {tp:,}")

In [ ]:
# Baseline confusion matrix visualization
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=baseline_confusion_matrix,
    display_labels=["No conversion", "Conversion"],
).plot(
    ax=ax,
    values_format=",d",
    colorbar=False,
)

ax.set_title("Baseline Logistic Regression — Confusion Matrix")

plt.tight_layout()

figure_path = FIGURES_PATH / "baseline_logistic_confusion_matrix.png"
plt.savefig(figure_path, dpi=300, bbox_inches="tight")

plt.show()

print(f"Figure saved to: {figure_path}")

In [ ]:
# Baseline error diagnostics
# ---------------------------------------------------------------------------

actual_conversions = tp + fn
predicted_conversions = tp + fp

missed_conversion_rate = fn / actual_conversions
false_discovery_rate = fp / predicted_conversions

print(
    "Actual conversions missed by the model: "
    f"{missed_conversion_rate:.2%}"
)

print(
    "Predicted conversions that are false positives: "
    f"{false_discovery_rate:.2%}"
)

### Baseline error analysis

On the internal test set, the baseline correctly identifies **1,246 of 1,836 actual conversions**, while **590 converters are classified as non-converters**. The model therefore misses approximately **32.1% of actual conversions**.

False-positive predictions are comparatively limited: **196 non-converting visitors** are incorrectly classified as converters. Approximately **13.6% of all predicted conversions** are false positives.

This error profile is consistent with the model's higher precision than recall. The baseline is relatively conservative when assigning the positive class, producing comparatively few false-positive conversion predictions at the cost of missing some actual converters.

These results motivate later investigation of class weighting and decision-threshold adjustment. However, any modification will be evaluated primarily on F1-score rather than recall alone, since improving recall at the expense of excessive false positives may not improve the competition metric.


## 3. Baseline Evaluation and Cross-Validation

Performance on a single train/test split can depend on the particular observations assigned to each subset. The baseline is therefore evaluated using **5-fold stratified cross-validation** on the training set.

Stratification preserves the class distribution within each fold, which is particularly important given the low conversion rate.

The complete pipeline is cross-validated rather than preprocessing the data beforehand. Consequently, preprocessing parameters are learned independently within each training fold, preventing information from the corresponding validation fold from leaking into model training.

The internal test set remains excluded from cross-validation.


In [ ]:
# Stratified cross-validation strategy
# ---------------------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

In [ ]:
# Baseline cross-validation
# ---------------------------------------------------------------------------

baseline_cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring="f1",
    n_jobs=-1,
)

print("Baseline F1 scores by fold:")
for fold, score in enumerate(baseline_cv_scores, start=1):
    print(f"Fold {fold}: {score:.4f}")

print(f"\nMean CV F1: {baseline_cv_scores.mean():.4f}")
print(f"CV F1 standard deviation: {baseline_cv_scores.std():.4f}")

### Cross-validation findings

The baseline logistic-regression pipeline achieves a mean **5-fold cross-validated F1-score of 0.7635**, with a standard deviation of **0.0046**.

Performance is consistent across folds, with F1-scores ranging from **0.7585 to 0.7705**. The cross-validation mean is also very close to both the training F1-score (**0.7637**) and the internal-test F1-score (**0.7602**).

This consistency indicates that the baseline model is stable across different subsets of the training data and shows no substantial evidence of overfitting.

The baseline therefore provides a reliable reference point for subsequent optimization. Improvements should be assessed against the cross-validated F1-score rather than against a single train/test result alone.


In [ ]:
# Baseline model summary
# ---------------------------------------------------------------------------

model_results = pd.DataFrame(
    [
        {
            "Model": "Baseline Logistic Regression",
            "CV F1 Mean": baseline_cv_scores.mean(),
            "CV F1 Std": baseline_cv_scores.std(),
            "Train F1": f1_score(y_train, y_train_pred),
            "Internal Test F1": f1_score(y_test, y_test_pred),
            "Test Precision": precision_score(y_test, y_test_pred),
            "Test Recall": recall_score(y_test, y_test_pred),
        }
    ]
)

model_results.round(4)

## 4. Logistic-Regression Optimization

The baseline model is stable but exhibits higher precision than recall, suggesting that its default classification behavior is relatively conservative.

A focused hyperparameter search is therefore used to evaluate two aspects of logistic regression:

* **regularization strength (`C`)**, which controls the degree of coefficient regularization;
* **class weighting**, which tests whether giving greater importance to the minority conversion class improves F1-score.

The search is intentionally limited to a small set of meaningful configurations. Hyperparameters are selected using 5-fold stratified cross-validation with **F1-score** as the optimization metric.

The internal test set remains excluded from hyperparameter selection.


In [ ]:
# Logistic-regression hyperparameter grid
# ---------------------------------------------------------------------------

logistic_param_grid = {
    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0,
    ],
    "classifier__class_weight": [
        None,
        "balanced",
    ],
}

In [ ]:
# Focused logistic-regression search
# ---------------------------------------------------------------------------

logistic_grid_search = GridSearchCV(
    estimator=baseline_model,
    param_grid=logistic_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

logistic_grid_search.fit(X_train, y_train)

print(
    "Best CV F1: "
    f"{logistic_grid_search.best_score_:.4f}"
)

print("\nBest parameters:")
print(logistic_grid_search.best_params_)

In [ ]:
# Hyperparameter search results
# ---------------------------------------------------------------------------

logistic_search_results = pd.DataFrame(
    logistic_grid_search.cv_results_
)

logistic_search_summary = (
    logistic_search_results[
        [
            "param_classifier__C",
            "param_classifier__class_weight",
            "mean_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .rename(
        columns={
            "param_classifier__C": "C",
            "param_classifier__class_weight": "Class Weight",
            "mean_train_score": "Train F1 Mean",
            "mean_test_score": "CV F1 Mean",
            "std_test_score": "CV F1 Std",
            "rank_test_score": "Rank",
        }
    )
    .sort_values("Rank")
)

logistic_search_summary.round(4)

In [ ]:
# Improve search-results presentation
# ---------------------------------------------------------------------------

logistic_search_summary["Class Weight"] = (
    logistic_search_summary["Class Weight"]
    .fillna("None")
)

logistic_search_summary.round(4)

### Hyperparameter search interpretation

The search results show that logistic-regression performance is relatively insensitive to moderate changes in regularization strength. Among the unweighted configurations, mean cross-validated F1 increases from **0.7548 at `C=0.01`** to **0.7640 at `C=10.0`**, with only a marginal improvement over the baseline `C=1.0` configuration (**0.7635**).

Applying `class_weight="balanced"` substantially reduces F1-score across every tested regularization strength, with cross-validated scores of approximately **0.511**. Although class weighting increases the relative importance assigned to the minority class during training, these results demonstrate that it does not improve the competition's F1 objective for this dataset.

Training and validation scores remain closely aligned across the tested configurations, providing no indication of substantial overfitting.

The best tested logistic-regression configuration is therefore **`C=10.0` with no class weighting**, achieving a mean cross-validated F1-score of **0.7640 ± 0.0039**. However, its improvement over the baseline is very small, so further gains are more likely to come from the decision rule, targeted feature representation, or nonlinear modeling than from additional regularization tuning.


In [ ]:
# Best logistic-regression configuration
# ---------------------------------------------------------------------------

tuned_logistic_model = logistic_grid_search.best_estimator_

print(tuned_logistic_model)

In [ ]:
# Out-of-fold conversion probabilities
# ---------------------------------------------------------------------------

oof_conversion_probabilities = cross_val_predict(
    tuned_logistic_model,
    X_train,
    y_train,
    cv=cv_strategy,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

print(
    "OOF probabilities generated:",
    len(oof_conversion_probabilities),
)

In [ ]:
# F1-score across candidate decision thresholds
# ---------------------------------------------------------------------------

thresholds = np.arange(0.10, 0.81, 0.01)

threshold_results = []

for threshold in thresholds:
    oof_predictions = (
        oof_conversion_probabilities >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "Threshold": threshold,
            "F1": f1_score(
                y_train,
                oof_predictions,
            ),
            "Precision": precision_score(
                y_train,
                oof_predictions,
            ),
            "Recall": recall_score(
                y_train,
                oof_predictions,
            ),
        }
    )

threshold_results = pd.DataFrame(threshold_results)

best_threshold_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

threshold_results.sort_values(
    "F1",
    ascending=False,
).head(10).round(4)

### Decision-threshold optimization

The default logistic-regression decision threshold of `0.50` is not necessarily optimal for F1-score, particularly given the strong class imbalance in the conversion target.

Using out-of-fold predicted probabilities from the training data, candidate thresholds were evaluated without using the internal test set. This preserves the internal test set for unbiased final evaluation.

The best tested threshold is **0.43**, producing an out-of-fold **F1-score of 0.7712**, with **precision of 0.8244** and **recall of 0.7245**.

Compared with the default decision rule, lowering the threshold increases recall while accepting a moderate reduction in precision. The resulting balance produces a higher F1-score.

Several nearby thresholds achieve very similar results, indicating that performance is relatively stable around the optimum rather than being dependent on a single precise cutoff. The value `0.43` is retained as the best threshold from the predefined search grid and will remain fixed for subsequent evaluation.


In [ ]:
# Threshold-performance visualization
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    label="F1",
)
ax.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    label="Precision",
)
ax.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    label="Recall",
)

ax.axvline(
    best_threshold_row["Threshold"],
    linestyle="--",
    label=f"Selected threshold ({best_threshold_row['Threshold']:.2f})",
)

ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title("Logistic Regression — Decision Threshold Performance")
ax.legend()

plt.tight_layout()

figure_path = FIGURES_PATH / "logistic_threshold_performance.png"
plt.savefig(figure_path, dpi=300, bbox_inches="tight")

plt.show()

print(f"Figure saved to: {figure_path}")

In [ ]:
# Selected training-only decision threshold
# ---------------------------------------------------------------------------

SELECTED_THRESHOLD = float(
    best_threshold_row["Threshold"]
)

print(f"Selected threshold: {SELECTED_THRESHOLD:.2f}")

## 5. Targeted Feature Engineering

The exploratory analysis identified several relationships that may not be represented fully by a purely additive linear decision function.

In particular:

* conversion probability changes strongly with `total_pages_visited`;
* the relationship between age and conversion may not be strictly linear;
* returning and new users exhibit substantially different conversion rates, suggesting that visitor status may interact with browsing behavior.

A small set of hypothesis-driven features is therefore evaluated. The objective is not to maximize the number of predictors, but to determine whether additional nonlinear and interaction information produces a meaningful improvement in out-of-fold F1-score.

The original predictors are retained, and engineered features will only be kept if they improve predictive performance consistently.


pages_squared
→ allows curvature in the pages/conversion relationship

age_squared
→ allows a nonlinear age effect

new_user_pages
→ allows browsing behavior to have a different effect for new vs returning users

In [ ]:
# Polynomial feature groups
# ---------------------------------------------------------------------------

polynomial_features = [
    "age",
    "total_pages_visited",
    "new_user",
]

categorical_features = [
    "country",
    "source",
]


# Polynomial preprocessing
# ---------------------------------------------------------------------------

polynomial_preprocessor = ColumnTransformer(
    transformers=[
        (
            "polynomial",
            Pipeline(
                steps=[
                    (
                        "polynomial_features",
                        PolynomialFeatures(
                            degree=2,
                            include_bias=False,
                        ),
                    ),
                    (
                        "scaler",
                        StandardScaler(),
                    ),
                ]
            ),
            polynomial_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)

In [ ]:
# Polynomial logistic-regression pipeline
# ---------------------------------------------------------------------------

polynomial_logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            polynomial_preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                C=10.0,
                class_weight=None,
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

polynomial_logistic_model

In [ ]:
# Inspect generated polynomial terms
# ---------------------------------------------------------------------------

polynomial_transformer = PolynomialFeatures(
    degree=2,
    include_bias=False,
)

polynomial_transformer.fit(
    X_train[polynomial_features]
)

generated_polynomial_features = (
    polynomial_transformer.get_feature_names_out(
        polynomial_features
    )
)

print("Generated polynomial features:")

for feature in generated_polynomial_features:
    print(f"- {feature}")

In [ ]:
# Polynomial logistic-regression cross-validation
# ---------------------------------------------------------------------------

polynomial_cv_scores = cross_val_score(
    polynomial_logistic_model,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring="f1",
    n_jobs=-1,
)

print("Polynomial logistic F1 scores by fold:")

for fold, score in enumerate(
    polynomial_cv_scores,
    start=1,
):
    print(f"Fold {fold}: {score:.4f}")

print(
    f"\nMean CV F1: "
    f"{polynomial_cv_scores.mean():.4f}"
)

print(
    f"CV F1 standard deviation: "
    f"{polynomial_cv_scores.std():.4f}"
)

print(
    "Change vs tuned logistic: "
    f"{polynomial_cv_scores.mean() - logistic_grid_search.best_score_:+.4f}"
)

### Feature-engineering results

The polynomial logistic-regression model achieves a mean cross-validated **F1-score of 0.7630 ± 0.0051**, compared with **0.7640 ± 0.0039** for the tuned logistic regression using the original predictors.

The polynomial representation includes second-order terms and pairwise interactions among `age`, `total_pages_visited`, and `new_user`, allowing the linear classifier to represent relationships such as nonlinear age and browsing effects as well as interactions between visitor characteristics.

Despite this additional flexibility, cross-validated F1 decreases by approximately **0.0009**, while variability across folds increases slightly. The additional features therefore provide no evidence of improved generalization.

The polynomial features are consequently **not retained**. The tuned logistic regression using the original predictors remains the preferred linear specification because it achieves slightly better performance with lower complexity.

This experiment indicates that manually increasing the flexibility of the logistic-regression feature representation does not materially improve predictive performance. The next step is therefore to evaluate whether a nonlinear classifier can capture additional structure directly.


## 6. Nonlinear Challenger

The logistic-regression experiments provide stable predictive performance, while polynomial feature engineering does not produce a meaningful improvement.

A **Random Forest classifier** is therefore evaluated as a single nonlinear challenger. Unlike logistic regression, Random Forest can capture nonlinear relationships and interactions directly without requiring these patterns to be specified manually.

This is particularly relevant given the strong nonlinear association observed between `total_pages_visited` and conversion during exploratory analysis.

The challenger uses the original five predictors rather than the rejected polynomial features. Categorical variables are one-hot encoded, while numerical and binary variables are passed through without scaling because tree-based models do not require standardized feature scales.

The objective is not to conduct an exhaustive search across multiple tree-based algorithms, but to determine whether a reasonably configured nonlinear model provides a meaningful improvement over the tuned logistic-regression benchmark.

### Why Random Forest?

A **Random Forest classifier** is selected as the nonlinear challenger.

This choice is motivated by several characteristics of the dataset and the exploratory analysis:

- the dataset contains a relatively small number of predictors but a large number of observations;
- `total_pages_visited` shows a strong nonlinear association with conversion;
- conversion behavior may depend on interactions between browsing activity, user status, age, country, and acquisition source;
- Random Forest can capture nonlinear relationships and feature interactions without requiring them to be specified manually;
- tree-based models do not require numerical features to be standardized;
- Random Forest provides feature-importance measures that can support interpretation of the final comparison;
- the model provides a clear methodological contrast with logistic regression while remaining straightforward to explain.


### Alternative nonlinear models

Several other classifiers could reasonably be considered.

A **Decision Tree** would provide high interpretability and capture nonlinear relationships, but a single tree is generally more sensitive to the training sample and more prone to overfitting than an ensemble.

**Gradient Boosting** could also be appropriate because sequentially fitted trees can capture complex nonlinear patterns and interactions. More advanced implementations such as XGBoost, LightGBM, or CatBoost could potentially provide strong predictive performance, but they would introduce additional model complexity, tuning requirements, and dependencies that are not necessary for establishing whether nonlinear modeling adds value in this project.

A **Support Vector Machine** with a nonlinear kernel could model nonlinear decision boundaries, but it is less attractive for a dataset of this size because of computational cost and reduced interpretability.

A **K-Nearest Neighbors** classifier could represent local nonlinear patterns, but prediction can become computationally expensive on a large dataset and its distance-based formulation is less naturally suited to the mixture of numerical, categorical, and binary predictors.

### Challenger strategy

Random Forest therefore provides a suitable balance between nonlinear modeling capacity, robustness, computational feasibility, and interpretability.

The objective is not to train every available classification algorithm. Instead, one justified nonlinear challenger is evaluated against the established logistic-regression benchmark. Further complexity would only be warranted if the challenger demonstrates a meaningful and stable improvement in cross-validated F1-score.

### 6.1 Random-forest

In [ ]:
# Random-forest preprocessing
# ---------------------------------------------------------------------------

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            categorical_features,
        ),
        (
            "numerical",
            "passthrough",
            numerical_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
)

In [ ]:
# Random-forest challenger
# ---------------------------------------------------------------------------

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor,
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                min_samples_leaf=5,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_model

Why these settings?

n_estimators=200 gives us a sufficiently stable ensemble without making this experiment unnecessarily expensive. min_samples_leaf=5 adds modest regularization and helps prevent the forest from creating leaves based on extremely small groups of observations.

I deliberately leave class_weight=None rather than assuming class balancing is beneficial merely because the target is imbalanced. Logistic experiment already demonstrated why imbalance treatments need empirical validation.

Most importantly, this is a challenger, not yet a heavily tuned model.

In [ ]:
# Random-forest cross-validation
# ---------------------------------------------------------------------------

random_forest_cv_scores = cross_val_score(
    random_forest_model,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring="f1",
    n_jobs=-1,
)

print("Random Forest F1 scores by fold:")

for fold, score in enumerate(
    random_forest_cv_scores,
    start=1,
):
    print(f"Fold {fold}: {score:.4f}")

print(
    f"\nMean CV F1: "
    f"{random_forest_cv_scores.mean():.4f}"
)

print(
    f"CV F1 standard deviation: "
    f"{random_forest_cv_scores.std():.4f}"
)

print(
    "Change vs tuned logistic: "
    f"{random_forest_cv_scores.mean() - logistic_grid_search.best_score_:+.4f}"
)

### Initial Random Forest results

The initial Random Forest challenger achieves a mean cross-validated **F1-score of 0.7532 ± 0.0052**.

This is approximately **0.0107 below** the tuned logistic-regression benchmark of **0.7640 ± 0.0039**. Performance is also slightly more variable across folds.

The result provides no evidence that the initial nonlinear model improves predictive performance. However, Random Forest behavior can depend meaningfully on tree depth and minimum leaf size. Before rejecting the nonlinear challenger, a small targeted hyperparameter search is performed to determine whether modest regularization changes improve generalization.

The search remains intentionally limited because the objective is to evaluate one serious nonlinear alternative rather than conduct an exhaustive ensemble-model optimization.

In [ ]:
# Focused Random Forest parameter grid
# ---------------------------------------------------------------------------

random_forest_param_grid = {
    "classifier__max_depth": [
        None,
        10,
        20,
    ],
    "classifier__min_samples_leaf": [
        2,
        5,
        10,
    ],
}

In [ ]:
# Focused Random Forest search
# ---------------------------------------------------------------------------

random_forest_grid_search = GridSearchCV(
    estimator=random_forest_model,
    param_grid=random_forest_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

random_forest_grid_search.fit(
    X_train,
    y_train,
)

print(
    f"Best Random Forest CV F1: "
    f"{random_forest_grid_search.best_score_:.4f}"
)

print("\nBest parameters:")
print(random_forest_grid_search.best_params_)

print(
    "\nChange vs tuned logistic: "
    f"{random_forest_grid_search.best_score_ - logistic_grid_search.best_score_:+.4f}"
)

In [ ]:
# Random Forest search results
# ---------------------------------------------------------------------------

random_forest_search_results = pd.DataFrame(
    random_forest_grid_search.cv_results_
)

random_forest_search_summary = (
    random_forest_search_results[
        [
            "param_classifier__max_depth",
            "param_classifier__min_samples_leaf",
            "mean_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .rename(
        columns={
            "param_classifier__max_depth": "Max Depth",
            "param_classifier__min_samples_leaf": "Min Samples Leaf",
            "mean_train_score": "Train F1 Mean",
            "mean_test_score": "CV F1 Mean",
            "std_test_score": "CV F1 Std",
            "rank_test_score": "Rank",
        }
    )
    .sort_values("Rank")
)

random_forest_search_summary.round(4)

### Random Forest optimization result

The focused Random Forest search improves mean cross-validated F1 from **0.7532** to **0.7603**.

The selected configuration uses a maximum tree depth of **10** and a minimum of **5 observations per leaf**, indicating that restricting tree complexity improves generalization relative to the initial unrestricted-depth forest.

Despite this improvement, the optimized Random Forest remains approximately **0.0037 F1 below** the tuned logistic-regression model (**0.7640**). The nonlinear challenger therefore becomes more competitive after regularization, but it does not demonstrate superior cross-validated predictive performance.

The complete search results are examined before making the final model-selection decision.

### 6.2 Gradient Boosting Challenger

The optimized Random Forest substantially improves upon its initial configuration, but its mean cross-validated F1-score remains below the tuned logistic-regression benchmark.

A second tree-based approach is therefore evaluated to determine whether the lack of improvement is specific to Random Forest or whether the simpler logistic model already captures most of the available predictive signal.

A **Histogram-based Gradient Boosting classifier** is selected for this experiment.

Unlike Random Forest, which builds many trees independently and averages their predictions, gradient boosting builds trees sequentially. Each stage attempts to improve upon the errors made by the preceding ensemble. This can allow boosting to capture complex decision boundaries and nonlinear relationships more efficiently.

This is particularly relevant for the current dataset because exploratory analysis identified a strong nonlinear relationship between `total_pages_visited` and conversion, together with potential interactions between browsing behavior and visitor characteristics.

`HistGradientBoostingClassifier` is preferred over the traditional `GradientBoostingClassifier` because it is designed to train efficiently on larger datasets.

The objective remains focused: first establish whether gradient boosting provides a meaningful cross-validated improvement. Hyperparameter and decision-threshold optimization will only be justified if the initial model is competitive.

Preprocessing difference

HistGradientBoostingClassifier expects dense input, whereas our existing OneHotEncoder may produce a sparse matrix. We therefore create a dedicated preprocessing pipeline.

Numerical features still do not need scaling for this tree-based model.

In [ ]:
# Gradient-boosting preprocessing
# ---------------------------------------------------------------------------

boosting_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
        (
            "numerical",
            "passthrough",
            numerical_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
)

In [ ]:
# Initial Histogram Gradient Boosting challenger
# ---------------------------------------------------------------------------

gradient_boosting_model = Pipeline(
    steps=[
        (
            "preprocessor",
            boosting_preprocessor,
        ),
        (
            "classifier",
            HistGradientBoostingClassifier(
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

gradient_boosting_model

In [ ]:
# Initial Gradient Boosting cross-validation
# ---------------------------------------------------------------------------

gradient_boosting_cv_scores = cross_val_score(
    gradient_boosting_model,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring="f1",
    n_jobs=-1,
)

print("Gradient Boosting F1 scores by fold:")

for fold, score in enumerate(
    gradient_boosting_cv_scores,
    start=1,
):
    print(f"Fold {fold}: {score:.4f}")

print(
    f"\nMean CV F1: "
    f"{gradient_boosting_cv_scores.mean():.4f}"
)

print(
    f"CV F1 standard deviation: "
    f"{gradient_boosting_cv_scores.std():.4f}"
)

print(
    "Change vs tuned logistic: "
    f"{gradient_boosting_cv_scores.mean() - logistic_grid_search.best_score_:+.4f}"
)

print(
    "Change vs optimized Random Forest: "
    f"{gradient_boosting_cv_scores.mean() - random_forest_grid_search.best_score_:+.4f}"
)

#### Initial Gradient Boosting results

The initial Histogram Gradient Boosting model achieves a mean cross-validated **F1-score of 0.7605 ± 0.0042**.

This slightly exceeds the optimized Random Forest result of **0.7603**, while remaining approximately **0.0035 below** the tuned logistic-regression benchmark of **0.7640**.

The relatively small gap and stable performance across folds suggest that Gradient Boosting remains a credible nonlinear candidate. Unlike the Random Forest experiment, this model has not yet received any model-specific tuning.

A small targeted hyperparameter search is therefore justified before deciding whether the boosting approach should be rejected or retained for further optimization.

### Focused Gradient Boosting optimization

For this model I would tune only three meaningful controls:

learning_rate controls how aggressively each boosting stage updates the ensemble. max_leaf_nodes controls tree complexity. l2_regularization penalizes overly complex solutions.

In [ ]:
# Focused Gradient Boosting parameter grid
# ---------------------------------------------------------------------------

gradient_boosting_param_grid = {
    "classifier__learning_rate": [
        0.05,
        0.10,
    ],
    "classifier__max_leaf_nodes": [
        15,
        31,
        63,
    ],
    "classifier__l2_regularization": [
        0.0,
        1.0,
    ],
}

That gives us 12 configurations × 5 folds = 60 fits.

In [ ]:
# Focused Gradient Boosting search
# ---------------------------------------------------------------------------

gradient_boosting_grid_search = GridSearchCV(
    estimator=gradient_boosting_model,
    param_grid=gradient_boosting_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

gradient_boosting_grid_search.fit(
    X_train,
    y_train,
)

print(
    f"Best Gradient Boosting CV F1: "
    f"{gradient_boosting_grid_search.best_score_:.4f}"
)

print("\nBest parameters:")
print(
    gradient_boosting_grid_search.best_params_
)

print(
    "\nChange vs tuned logistic: "
    f"{gradient_boosting_grid_search.best_score_ - logistic_grid_search.best_score_:+.4f}"
)

print(
    "Change vs optimized Random Forest: "
    f"{gradient_boosting_grid_search.best_score_ - random_forest_grid_search.best_score_:+.4f}"
)

#### Focused Gradient Boosting optimization result

The focused hyperparameter search improves Gradient Boosting from a mean cross-validated F1-score of **0.7605** to **0.7650**.

The selected configuration uses:

- `learning_rate=0.10`
- `max_leaf_nodes=15`
- `l2_regularization=1.0`

The preference for fewer leaf nodes and positive L2 regularization suggests that a moderately constrained boosting model generalizes better than more flexible alternatives.

The optimized model slightly exceeds the tuned logistic-regression benchmark of **0.7640** by approximately **0.0010 F1**, and exceeds the optimized Random Forest by approximately **0.0047**.

Although the improvement over logistic regression is small, Gradient Boosting is now sufficiently competitive to justify decision-threshold optimization.

To ensure a fair comparison with logistic regression, the threshold is optimized using out-of-fold probabilities generated exclusively from the training data. The internal test set remains untouched.

In [ ]:
# Best Gradient Boosting estimator
# ---------------------------------------------------------------------------

tuned_gradient_boosting_model = (
    gradient_boosting_grid_search.best_estimator_
)

In [ ]:
# Out-of-fold probabilities for threshold optimization
# ---------------------------------------------------------------------------

gradient_boosting_oof_probabilities = cross_val_predict(
    tuned_gradient_boosting_model,
    X_train,
    y_train,
    cv=cv_strategy,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

print(
    "OOF probabilities generated:",
    len(gradient_boosting_oof_probabilities),
)

In [ ]:
# Gradient Boosting threshold search
# ---------------------------------------------------------------------------

boosting_threshold_results = []

for threshold in np.arange(0.10, 0.81, 0.01):
    predictions = (
        gradient_boosting_oof_probabilities >= threshold
    ).astype(int)

    boosting_threshold_results.append(
        {
            "Threshold": threshold,
            "F1": f1_score(
                y_train,
                predictions,
            ),
            "Precision": precision_score(
                y_train,
                predictions,
            ),
            "Recall": recall_score(
                y_train,
                predictions,
            ),
        }
    )

boosting_threshold_results = pd.DataFrame(
    boosting_threshold_results
)

boosting_threshold_results = (
    boosting_threshold_results
    .sort_values(
        "F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

boosting_threshold_results.head(10).round(4)

In [ ]:
# Select best Gradient Boosting threshold
# ---------------------------------------------------------------------------

best_boosting_threshold_row = (
    boosting_threshold_results.iloc[0]
)

BOOSTING_SELECTED_THRESHOLD = float(
    best_boosting_threshold_row["Threshold"]
)

print(
    f"Selected Gradient Boosting threshold: "
    f"{BOOSTING_SELECTED_THRESHOLD:.2f}"
)

print(
    f"OOF F1: "
    f"{best_boosting_threshold_row['F1']:.4f}"
)

print(
    f"OOF precision: "
    f"{best_boosting_threshold_row['Precision']:.4f}"
)

print(
    f"OOF recall: "
    f"{best_boosting_threshold_row['Recall']:.4f}"
)

#### Gradient Boosting threshold optimization

Decision-threshold optimization is performed using out-of-fold probabilities generated exclusively from the training data.

The highest OOF F1-score is obtained at a threshold of **0.43**, producing:

- **F1-score:** 0.7704
- **Precision:** 0.8135
- **Recall:** 0.7315

Lowering the threshold from the default `0.50` increases recall at the cost of some precision, producing a better balance for the F1 objective.

Several neighboring thresholds produce nearly identical results, indicating that performance is relatively stable around the selected operating point rather than depending on a single highly specific threshold.

Despite the improvement from threshold optimization, the optimized Gradient Boosting model does not exceed the threshold-optimized logistic regression, which achieves an OOF F1-score of **0.7712**.

Gradient Boosting therefore provides no meaningful predictive advantage after both models are evaluated under comparable threshold-optimization procedures.

#### Threshold-performance comparison

To compare the two strongest candidate models under the same decision framework, their out-of-fold F1-scores are evaluated across the same range of classification thresholds.

This comparison separates two aspects of model performance:

1. the quality of the probability estimates produced by each classifier;
2. the decision threshold used to convert probabilities into binary predictions.

Both models reach their strongest F1 performance below the conventional `0.50` threshold, confirming that the default classification threshold is not optimal for this imbalanced conversion problem.

In [ ]:
# Threshold-performance comparison
# ---------------------------------------------------------------------------

logistic_threshold_plot = (
    threshold_results
    .sort_values("Threshold")
)

boosting_threshold_plot = (
    boosting_threshold_results
    .sort_values("Threshold")
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    logistic_threshold_plot["Threshold"],
    logistic_threshold_plot["F1"],
    label="Logistic Regression",
    linewidth=2,
)

ax.plot(
    boosting_threshold_plot["Threshold"],
    boosting_threshold_plot["F1"],
    label="Gradient Boosting",
    linewidth=2,
)

ax.axvline(
    SELECTED_THRESHOLD,
    linestyle="--",
    alpha=0.7,
    label=f"Selected threshold = {SELECTED_THRESHOLD:.2f}",
)

ax.set_title(
    "Out-of-Fold F1 Score Across Classification Thresholds"
)
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("F1 Score")

ax.legend()
ax.grid(alpha=0.2)

fig.tight_layout()

figure_path = (
    FIGURES_PATH
    / "model_threshold_comparison.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Figure saved to: {figure_path}")

The threshold-performance curves show that Logistic Regression and Gradient Boosting behave similarly across a broad range of decision thresholds.

Both models reach their strongest F1 performance around a threshold of **0.43**, rather than at the conventional `0.50` threshold.

The threshold-optimized Logistic Regression reaches an OOF F1-score of **0.7712**, compared with **0.7704** for Gradient Boosting. The difference is very small, indicating that the two models provide comparable predictive performance once their decision thresholds are optimized consistently.

Gradient Boosting achieves slightly higher recall at its selected threshold, whereas Logistic Regression achieves higher precision and a marginally higher overall F1-score.

Because the more complex Gradient Boosting model does not demonstrate a meaningful improvement over Logistic Regression, the simpler model remains preferable for final evaluation. It provides comparable predictive performance while offering greater transparency and easier interpretation.

## 7. Model Comparison and Selection

Model development is now complete.

The experiments progressed from a simple logistic-regression baseline to focused regularization tuning, decision-threshold optimization, targeted polynomial feature engineering, and nonlinear tree-based challengers.

Model selection is based primarily on **F1-score**, while also considering cross-validation stability, model complexity, and interpretability.

Importantly, the optimized models have not been selected based on their performance on the internal test set. Hyperparameters, feature-engineering decisions, and classification thresholds were determined exclusively from the training data using cross-validation or out-of-fold predictions.

### 7.1 Development comparison

In [ ]:
# Final model-development comparison
# ---------------------------------------------------------------------------

model_comparison = pd.DataFrame(
    [
        {
            "Model": "Baseline Logistic Regression",
            "Optimization": "None",
            "Evaluation": "5-fold CV",
            "F1": baseline_cv_scores.mean(),
            "F1 Std": baseline_cv_scores.std(),
        },
        {
            "Model": "Tuned Logistic Regression",
            "Optimization": "C=10",
            "Evaluation": "5-fold CV",
            "F1": logistic_grid_search.best_score_,
            "F1 Std": logistic_grid_search.cv_results_[
                "std_test_score"
            ][logistic_grid_search.best_index_],
        },
        {
            "Model": "Polynomial Logistic Regression",
            "Optimization": "Degree-2 terms",
            "Evaluation": "5-fold CV",
            "F1": polynomial_cv_scores.mean(),
            "F1 Std": polynomial_cv_scores.std(),
        },
        {
            "Model": "Optimized Random Forest",
            "Optimization": "Focused grid search",
            "Evaluation": "5-fold CV",
            "F1": random_forest_grid_search.best_score_,
            "F1 Std": random_forest_grid_search.cv_results_[
                "std_test_score"
            ][random_forest_grid_search.best_index_],
        },
        {
            "Model": "Optimized Gradient Boosting",
            "Optimization": "Focused grid search",
            "Evaluation": "5-fold CV",
            "F1": gradient_boosting_grid_search.best_score_,
            "F1 Std": gradient_boosting_grid_search.cv_results_[
                "std_test_score"
            ][gradient_boosting_grid_search.best_index_],
        },
    ]
)

model_comparison = model_comparison.sort_values(
    "F1",
    ascending=False,
).reset_index(drop=True)

model_comparison.round(4)

### 7.2 Finalist threshold comparison

In [ ]:
# Threshold-optimized finalist comparison
# ---------------------------------------------------------------------------

finalist_comparison = pd.DataFrame(
    [
        {
            "Model": "Tuned Logistic Regression",
            "Threshold": SELECTED_THRESHOLD,
            "OOF F1": best_threshold_row["F1"],
            "OOF Precision": best_threshold_row["Precision"],
            "OOF Recall": best_threshold_row["Recall"],
        },
        {
            "Model": "Optimized Gradient Boosting",
            "Threshold": BOOSTING_SELECTED_THRESHOLD,
            "OOF F1": best_boosting_threshold_row["F1"],
            "OOF Precision": best_boosting_threshold_row["Precision"],
            "OOF Recall": best_boosting_threshold_row["Recall"],
        },
    ]
)

finalist_comparison.round(4)

### 7.3 Final Model Selection

The final candidate comparison shows that the two strongest models achieve nearly identical predictive performance after decision-threshold optimization.

Tuned Logistic Regression reaches an out-of-fold F1-score of **0.7712**, compared with **0.7704** for optimized Gradient Boosting. Gradient Boosting provides slightly higher recall, whereas Logistic Regression provides higher precision and a marginally higher F1-score.

Given the negligible performance difference, the additional complexity of Gradient Boosting does not provide a meaningful predictive advantage.

The selected model is therefore **Logistic Regression with `C=10`, no class weighting, and a classification threshold of `0.43`**.

This model is retained because it combines:

- strong and stable cross-validated performance;
- the highest threshold-optimized OOF F1-score among the finalists;
- substantially greater simplicity and interpretability;
- no evidence that additional polynomial or nonlinear complexity produces a meaningful improvement.

All model, hyperparameter, feature-engineering, and threshold decisions were made using the training data only. The internal test set has not been used to select among the optimized candidate models.

The selected modeling strategy is now fixed and can be evaluated once on the internal test set.

### 7.4 Final Internal Test Evaluation

The modeling strategy is now fixed. The selected Logistic Regression model is evaluated once on the untouched internal test set using the classification threshold of **0.43** selected from out-of-fold training predictions.

This evaluation provides the final estimate of generalization performance before retraining the selected model on all labeled observations for the competition submission.

In [ ]:
# Final internal test evaluation
# ---------------------------------------------------------------------------

final_model = logistic_grid_search.best_estimator_
FINAL_THRESHOLD = SELECTED_THRESHOLD

final_test_probabilities = final_model.predict_proba(
    X_test
)[:, 1]

final_test_predictions = (
    final_test_probabilities >= FINAL_THRESHOLD
).astype(int)

final_test_f1 = f1_score(
    y_test,
    final_test_predictions,
)

final_test_precision = precision_score(
    y_test,
    final_test_predictions,
)

final_test_recall = recall_score(
    y_test,
    final_test_predictions,
)

print(f"Final internal test F1:        {final_test_f1:.4f}")
print(f"Final internal test precision: {final_test_precision:.4f}")
print(f"Final internal test recall:    {final_test_recall:.4f}")

In [ ]:
# Final internal test confusion matrix
# ---------------------------------------------------------------------------

final_confusion_matrix = confusion_matrix(
    y_test,
    final_test_predictions,
)

tn, fp, fn, tp = final_confusion_matrix.ravel()

print("Final confusion matrix:")
print(final_confusion_matrix)

print(f"\nTrue negatives:  {tn:,}")
print(f"False positives: {fp:,}")
print(f"False negatives: {fn:,}")
print(f"True positives:  {tp:,}")

In [ ]:
# Final internal test confusion matrix
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=final_confusion_matrix,
    display_labels=["Not Converted", "Converted"],
).plot(
    ax=ax,
    values_format=",",
    colorbar=False,
)

ax.set_title(
    "Final Logistic Regression — Internal Test Set"
)

fig.tight_layout()

figure_path = (
    FIGURES_PATH
    / "final_logistic_confusion_matrix.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Figure saved to: {figure_path}")

#### Final evaluation results

The selected Logistic Regression model achieves an internal test **F1-score of 0.7640**, with **precision of 0.8273** and **recall of 0.7097**.

The confusion matrix contains:

- **54,808 true negatives**
- **272 false positives**
- **533 false negatives**
- **1,303 true positives**

Of the **1,836 actual converters** in the internal test set, the model correctly identifies approximately **71.0%**, while approximately **29.0%** are missed.

Among observations predicted as converters, approximately **82.7%** actually convert. The model therefore maintains relatively high precision while recovering a substantial majority of positive cases.

The final internal-test F1-score of **0.7640** is slightly below the threshold-optimized out-of-fold training estimate of **0.7712**, a difference of approximately **0.0072**. This relatively small gap suggests that the selected modeling strategy generalizes consistently to previously unseen observations.

The internal test result is treated as the final evaluation of the selected modeling strategy. No further model, hyperparameter, feature-engineering, or threshold decisions are made based on this result.

## 8. Conclusion

This notebook developed and evaluated a focused set of supervised classification models for predicting user conversion, using **F1-score as the primary selection metric**.

The initial Logistic Regression baseline already provided strong and stable performance. Regularization tuning produced only a marginal improvement, while class weighting substantially reduced F1-score. Optimizing the classification threshold using out-of-fold training predictions improved the balance between precision and recall, with a selected threshold of **0.43**.

Additional model complexity did not produce a meaningful predictive advantage. Degree-2 polynomial features failed to improve Logistic Regression, and the optimized Random Forest remained below the linear benchmark. Histogram Gradient Boosting was the strongest nonlinear challenger and slightly exceeded Logistic Regression at the default classification threshold, but after equivalent threshold optimization the two models achieved nearly identical performance.

The final selected specification is therefore:

- **Model:** Logistic Regression
- **Regularization:** `C=10`
- **Class weighting:** None
- **Classification threshold:** `0.43`
- **Threshold-optimized OOF F1:** `0.7712`
- **Internal test F1:** `0.7640`
- **Internal test precision:** `0.8273`
- **Internal test recall:** `0.7097`

On the untouched internal test set, the model correctly identifies **1,303 of 1,836 converters**, while producing **272 false-positive conversion predictions**. The relatively small difference between out-of-fold and internal-test performance supports the stability of the selected modeling strategy.

The internal test evaluation concludes model development. No further modeling decisions will be based on the internal test results.

The next notebook will retrain the fixed modeling pipeline on the complete cleaned labeled dataset, generate predictions for the separate competition test set, create the submission file, and translate the final model into interpretable business insights.